In [1]:
import sys
import os
import pandas as pd
from IPython.display import display, HTML
import keyring
from datetime import datetime

# Get the parent directory of the notebook (i.e., project root)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.automated_isone_data_update import automated_isone_data_update

# FLP database connection tools path
flp_db_tools_path = r"C:\Users\cbrooks\OneDrive - FIRSTLIGHTPOWER.COM\Documents\Python\flp_database_connection_tools"
database_helpers = os.path.join(flp_db_tools_path,"Helpers")
if database_helpers not in sys.path:
    sys.path.append(database_helpers)
from flp_database_connector import flp_database_connector

In [2]:
## Inputs
# Get token from Windows Credential Manager
token = keyring.get_password("pharos_api", "api_token")
USERNAME = r"cal.brooks@firstlight.energy"
tz = 'America/New_York'
current_year = datetime.today().year

## Schedule offers historic API (ISONE)

GET `/api/isone/schedule_offers/historic` — returns schedule offer price data as CSV.

**Parameters:** `organization_key`, `start_date` (required), `end_date` (optional, defaults to start_date), `market` (required: `day_ahead`, `reoffer`, or `real_time`). Dates in YYYY-MM-DD or relative: `yesterday`, `today`, `tomorrow`. No more than 365 days per request.

In [3]:
from src.pharos_ams_query import query_schedule_offers_historic, process_schedule_offers_historic

# Example: day-ahead schedule offer prices for a short date range
organization_key = "ho-fl"  # "FirstLight Power" = "hp", "FirstLight Power (50876)"="ho-fl". I think we want ho-fl
start_date = "2026-03-01"
end_date = "2026-03-05"
market = "day_ahead"  # or "reoffer", "real_time"

mapping_path = os.path.join(project_root, "data", "maps", "ISONE Location Mapping.csv")

df_schedule = query_schedule_offers_historic(
    token,
    organization_key=organization_key,
    start_date=start_date,
    market=market,
    end_date=end_date,
)
print(f"Retrieved {len(df_schedule)} rows (raw)")

df_schedule = process_schedule_offers_historic(df_schedule, mapping_path, tz=tz)
print(f"After post-processing: {len(df_schedule)} rows, {len(df_schedule.columns)} columns")
display(df_schedule.head())


Retrieved 4500 rows (raw)
After post-processing: 1440 rows, 28 columns


,date,hour_ending,market_type,mw_0,price_0,mw_1,price_1,mw_2,price_2,mw_3,...,price_7,mw_8,price_8,mw_9,price_9,name,asset,datetime_hb,service,ops_type
0,2026-03-01,1,day_ahead,2.0,-50.0,3.0,-49.99,NaN,NaN,NaN,...,None,None,None,None,None,BULLS BRIDGE,Bulls Bridge,2026-03-01 00:00:00-05:00,energy,generation
1,2026-03-01,2,day_ahead,2.0,-50.0,3.0,-49.99,NaN,NaN,NaN,...,None,None,None,None,None,BULLS BRIDGE,Bulls Bridge,2026-03-01 01:00:00-05:00,energy,generation
2,2026-03-01,3,day_ahead,2.0,-50.0,3.0,-49.99,NaN,NaN,NaN,...,None,None,None,None,None,BULLS BRIDGE,Bulls Bridge,2026-03-01 02:00:00-05:00,energy,generation
3,2026-03-01,4,day_ahead,2.0,-50.0,3.0,-49.99,NaN,NaN,NaN,...,None,None,None,None,None,BULLS BRIDGE,Bulls Bridge,2026-03-01 03:00:00-05:00,energy,generation
4,2026-03-01,5,day_ahead,2.0,-50.0,3.0,-49.99,NaN,NaN,NaN,...,None,None,None,None,None,BULLS BRIDGE,Bulls Bridge,2026-03-01 04:00:00-05:00,energy,generation


In [4]:
df_schedule.to_excel('test.xlsx')

In [ ]:
# Automatically refresh data in DAAS positions database
db_table_name = "ops.isone_hourly_ancillary"
mis_report = "SD_DAASCLEARED"
automated_isone_data_update(USERNAME, token, db_table_name, tz, mis_report, datetime(current_year,1,1),fill_with_zeros=True)

Using user-specified start_date: 2025-03-01
Checking for missing data from 2025-03-01 to 2026-02-02


c:\Users\cbrooks\OneDrive - FIRSTLIGHTPOWER.COM\Documents\Python\settlement_parsing_tools\src\automated_isone_data_update.py:249: UserWarning: WARNING: Merge test found 0 matches out of 5 expected records. This suggests a mismatch in composite columns. Sample existing datetime_he: [Timestamp('2025-03-01 01:00:00-0500', tz='America/New_York'), Timestamp('2025-03-01 01:00:00-0500', tz='America/New_York'), Timestamp('2025-03-01 01:00:00-0500', tz='America/New_York')], Sample expected datetime_he: [Timestamp('2025-03-01 02:00:00-0500', tz='America/New_York'), Timestamp('2025-03-01 02:00:00-0500', tz='America/New_York'), Timestamp('2025-03-01 02:00:00-0500', tz='America/New_York')]
  warnings.warn(


Found 5280 missing records across 6 dates
Missing dates: [datetime.date(2026, 1, 29), datetime.date(2026, 1, 30), datetime.date(2026, 1, 31), datetime.date(2026, 2, 1), datetime.date(2026, 2, 2), datetime.date(2026, 2, 3)]...
Grouped into 1 contiguous date ranges
Querying API 1/1: 2026-01-29 to 2026-02-04
  Retrieved 103 rows

Processing combined data from 1 API responses...
First few actual column names: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
First row sample: {0: 'FirstLight Power (50876)', 1: 'SD_DAASCLEARED', 2: '2026-01-29', 3: '2026-02-03 03:54:41 UTC', 4: 'C', 5: 'Firstlight Power Management LL', 6: None, 7: None, 8: None, 9: None}
Using only the first 13 columns and ignoring the rest.
0 duplicate rows removed based on most recent Version.
Processed 384 rows for upload.

Deduplicating against existing database records...


c:\Users\cbrooks\OneDrive - FIRSTLIGHTPOWER.COM\Documents\Python\settlement_parsing_tools\src\automated_isone_data_update.py:670: UserWarning: WARNING: Found 147 records where existing database value was 0, but API returned nonzero data. These will be updated.
Sample records being overwritten:
                 datetime_he         name    ops_type service da_volume
0  2026-01-29 01:00:00-05:00        CABOT  Generation   TMNSR         8
3  2026-01-29 01:00:00-05:00  ROCKY RIVER  Generation   TMNSR        20
4  2026-01-29 01:00:00-05:00        CABOT  Generation    TMOR         0
7  2026-01-29 01:00:00-05:00  ROCKY RIVER  Generation    TMOR         0
8  2026-01-29 01:00:00-05:00        CABOT  Generation    TMSR         0
11 2026-01-29 01:00:00-05:00  ROCKY RIVER  Generation    TMSR         0
15 2026-01-29 02:00:00-05:00  ROCKY RIVER  Generation   TMNSR        20
19 2026-01-29 02:00:00-05:00  ROCKY RIVER  Generation    TMOR         0
23 2026-01-29 02:00:00-05:00  ROCKY RIVER  Generation    

After deduplication: 384 rows remaining for upload.

Checking for any remaining missing data to fill with zeros...
Data range: 2026-01-29 01:00:00-05:00 to 2026-01-30 00:00:00-05:00

Filling 672 missing combinations with default values (0) for gaps between 2026-01-29 01:00:00-05:00 and 2026-01-30 00:00:00-05:00...
Added 672 records with default values.

Performing final deduplication check before upload...
No duplicates found in final data (count: 1056)
Converting volume columns to numeric types...
Uploading to database...
Target timezone for ops.isone_hourly_ancillary: America/New_York
Only datetime_he column was provided. datetime_hb column was created...
Beginning data upload of 1056 rows in update mode...
Data uploaded to database. Beginning merge with updates...
Update complete! Updated/inserted 1056 rows into ops.isone_hourly_ancillary
Upload complete!


In [ ]:
# Automatically refresh data in RT reserves positions database
db_table_name = "ops.isone_hourly_ancillary"
mis_report = "OI_UNITRTRSV"
automated_isone_data_update(USERNAME, token, db_table_name, tz, mis_report, datetime(current_year,1,1), fill_with_zeros=True)

Using user-specified start_date: 2025-03-01
Checking for missing data from 2025-03-01 to 2026-02-02


c:\Users\cbrooks\OneDrive - FIRSTLIGHTPOWER.COM\Documents\Python\settlement_parsing_tools\src\automated_isone_data_update.py:249: UserWarning: WARNING: Merge test found 0 matches out of 5 expected records. This suggests a mismatch in composite columns. Sample existing datetime_he: [Timestamp('2025-03-01 01:00:00-0500', tz='America/New_York'), Timestamp('2025-03-01 01:00:00-0500', tz='America/New_York'), Timestamp('2025-03-01 01:00:00-0500', tz='America/New_York')], Sample expected datetime_he: [Timestamp('2025-03-01 02:00:00-0500', tz='America/New_York'), Timestamp('2025-03-01 02:00:00-0500', tz='America/New_York'), Timestamp('2025-03-01 02:00:00-0500', tz='America/New_York')]
  warnings.warn(


Found 72 missing records across 2 dates
Missing dates: [datetime.date(2026, 2, 2), datetime.date(2026, 2, 3)]...
Segmented into 1 monthly chunks (max 30 days each) for OI_UNITRTRSV
Grouped into 1 contiguous date ranges
Querying API 1/1: 2026-02-02 to 2026-02-04
  Retrieved 12650 rows

Processing combined data from 1 API responses...
First few actual column names: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
First row sample: {0: 'FirstLight Power (50876)', 1: 'OI_UNITRTRSV', 2: '2026-02-04', 3: '2026-02-04 18:10:02 UTC', 4: 'C', 5: 'Firstlight Power Management LL', 6: None, 7: None, 8: None, 9: None}
Using only the first 10 columns and ignoring the rest.
[DEBUG process_rt_reserve_data] Data_vs_Header_Code value counts before filter:
Data_vs_Header_Code
D    12223
C      242
H      122
T       61
Name: count, dtype: int64
[DEBUG process_rt_reserve_data] Before merge with mapping: 1037 rows
[DEBUG process_rt_reserve_data] Unique Asset_IDs in data: 17
[DEBUG process_rt_reserve_data] Sample Asset_IDs: [

c:\Users\cbrooks\OneDrive - FIRSTLIGHTPOWER.COM\Documents\Python\settlement_parsing_tools\src\process_as_positions.py:388: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  agg_dict = {col: 'mean' for col in designation_cols}


After deduplication: 2435 rows remaining for upload.

Checking for any remaining missing data to fill with zeros...
Data range: 2026-02-02 01:00:00-05:00 to 2026-02-04 13:00:00-05:00

Filling 318 missing combinations with default values (0) for gaps between 2026-02-02 01:00:00-05:00 and 2026-02-04 13:00:00-05:00...
Added 318 records with default values.

Performing final deduplication check before upload...
No duplicates found in final data (count: 2753)
Converting volume columns to numeric types...
Uploading to database...
Target timezone for ops.isone_hourly_ancillary: America/New_York
Only datetime_he column was provided. datetime_hb column was created...
Beginning data upload of 2753 rows in update mode...
Data uploaded to database. Beginning merge with updates...
Update complete! Updated/inserted 2753 rows into ops.isone_hourly_ancillary
Upload complete!


In [ ]:
# Automatically refresh data in energy positions database
db_table_name = "ops.isone_hourly_energy"
mis_report = "SR_RTLOCSUM"
automated_isone_data_update(USERNAME, token, db_table_name, tz, mis_report, datetime(current_year,1,1), fill_with_zeros=True)

Using user-specified start_date: 2025-01-01
Checking for missing data from 2025-01-01 to 2026-02-02


c:\Users\cbrooks\OneDrive - FIRSTLIGHTPOWER.COM\Documents\Python\settlement_parsing_tools\src\automated_isone_data_update.py:330: UserWarning: WARNING: Merge test found 0 matches out of 5 expected records. This suggests a mismatch in composite columns. Sample existing datetime_he: [Timestamp('2025-01-01 01:00:00-0500', tz='America/New_York'), Timestamp('2025-01-01 01:00:00-0500', tz='America/New_York'), Timestamp('2025-01-01 01:00:00-0500', tz='America/New_York')], Sample expected datetime_he: [Timestamp('2025-01-01 02:00:00-0500', tz='America/New_York'), Timestamp('2025-01-01 02:00:00-0500', tz='America/New_York'), Timestamp('2025-01-01 02:00:00-0500', tz='America/New_York')]
  warnings.warn(


Found 1920 missing records across 5 dates
Missing dates: [datetime.date(2026, 1, 30), datetime.date(2026, 1, 31), datetime.date(2026, 2, 1), datetime.date(2026, 2, 2), datetime.date(2026, 2, 3)]...
Grouped into 1 contiguous date ranges
Querying API 1/1: 2026-01-30 to 2026-02-04
  Retrieved 1623 rows

Processing combined data from 1 API responses...
No duplicate rows found.
Processed 1440 rows for upload.

Deduplicating against existing database records...
After deduplication: 1440 rows remaining for upload.

Checking for any remaining missing data to fill with zeros...
Data range: 2026-01-30 01:00:00-05:00 to 2026-02-02 00:00:00-05:00
No missing records found between data range.

Performing final deduplication check before upload...
No duplicates found in final data (count: 1440)
Converting volume columns to numeric types...
Uploading to database...
Target timezone for ops.isone_hourly_energy: America/New_York
Only datetime_he column was provided. datetime_hb column was created...
Begi

In [ ]:
# # Set parameters for database query
# USERNAME = r"firstlightpower\cbrooks"
# db_table_name = "ops.isone_hourly_energy"

# # Query existing data in database
# db_conn = flp_database_connector(USERNAME)
# sql_query = f"""
#     SELECT *
#     FROM {db_table_name}
# """
# existing_data = db_conn.read_from_db("DataQuant01", "", sql_query)

# db_conn.upload_data_to_quant_db(
#                     table_name="ops.backup_isone_hourly_ancillary_2026_01_06",
#                     df=existing_data,
#                     tz='America/New_York',
#                     mode="create",
#                     skip_prompt=True
#                 )

Target timezone for ops.backup_isone_hourly_ancillary_2026_01_06: America/New_York
Successfully processed both datetime_he and datetime_hb columns...
Beginning upload to create a new table or overwrite an existing one...
Created new table ops.backup_isone_hourly_ancillary_2026_01_06 with 1629960 rows!


In [5]:
from datetime import datetime, timedelta

 # Set parameters for database query & upload
USERNAME = r"firstlightpower\cbrooks"
db_table_name = "ops.isone_hourly_ancillary"
tz = 'America/New_York'

# Query existing data in database
db_conn = flp_database_connector(USERNAME)
sql_query = f"""
    SELECT *
    FROM {db_table_name}
"""
existing_data = db_conn.read_from_db("DataQuant01", "", sql_query)

start_date = existing_data['datetime_he'].min().date()
        
# End date: end of day 2 days prior to today
end_date = (datetime.now().date() - timedelta(days=2))

# Create expected datetime range (hourly intervals)
expected_datetimes = pd.date_range(
    start=start_date,
    end=end_date + timedelta(days=1),  # Include full last day
    freq='h',
    tz=tz
)

In [17]:
# Convert to UTC first, then convert to target timezone
existing_data['datetime_he'] = pd.to_datetime(
    existing_data['datetime_he'], 
    utc=True
).dt.tz_convert('America/New_York')

# Start date: earliest date in existing data
start_date = existing_data['datetime_he'].min().date()

# End date: end of day 2 days prior to today
end_date = (datetime.now().date() - timedelta(days=2))

print(f"Checking for missing data from {start_date} to {end_date}")

# Create expected datetime range (hourly intervals) - timezone-aware
expected_datetimes = pd.date_range(
    start=start_date,
    end=end_date + timedelta(days=1),  # Include full last day
    freq='H',
    tz='America/New_York'  # Make timezone-aware
)

Checking for missing data from 2025-03-01 to 2025-11-24


C:\Users\cbrooks\AppData\Local\Temp\ipykernel_32712\2554004317.py:16: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  expected_datetimes = pd.date_range(


In [ ]:

print(range(40))

range(0, 40)


In [18]:
import pandas as pd
import datetime
import warnings
import re
from zoneinfo import ZoneInfo   # Python 3.9+; or use pytz if you prefer
from pandas.api.types import is_datetime64_any_dtype, is_string_dtype, infer_dtype
from pytz.exceptions import AmbiguousTimeError, NonExistentTimeError

TZ = "America/New_York"

check = pd.Series([datetime.datetime(2025,11,26,12, tzinfo=ZoneInfo(TZ)), datetime.datetime(2025,11,26,13, tzinfo=ZoneInfo(TZ))])

tests = {
    "naive_strings": {
        "input": pd.Series(["2025-11-26 12:00", "2025-11-26 13:00"]),
        "expect": "ok"   # localize to TZ, preserve wall-clock times
    },
    "naive_datetimes": {
        "input": pd.Series([datetime.datetime(2025,11,26,12), datetime.datetime(2025,11,26,13)]),
        "expect": "ok"
    },
    "tzaware_matching_timestamp": {
        "input":pd.Series([pd.Timestamp("2025-11-26 12:00", tz=TZ), pd.Timestamp("2025-11-26 13:00", tz=TZ)]),
        "expect": "ok"  # accept as-is because tz matches
    },
    "tzaware_matching_datetime_zoneinfo": {
        "input":pd.Series([datetime.datetime(2025,11,26,12, tzinfo=ZoneInfo(TZ)), datetime.datetime(2025,11,26,13, tzinfo=ZoneInfo(TZ))]),
        "expect": "ok"
    },
    "tzaware_different_tz": {
        "input": pd.Series([pd.Timestamp("2025-11-26 12:00", tz="UTC"), pd.Timestamp("2025-11-26 13:00", tz="UTC")]),
        "expect": "raise"
    },
    "string_offset_matching": {
        # On 2025-11-26 New_York is UTC-05:00 (EST). This string uses -05:00 so should match.
        "input": pd.Series(["2025-11-26T12:00:00-05:00", "2025-11-26T13:00:00-05:00"]),
        "expect": "ok"
    },
    "string_offset_not_matching": {
        "input": pd.Series(["2025-11-26T18:00:00+01:00", "2025-11-26T19:00:00+01:00"]),
        "expect": "ok" # This is a different timezone than the target timezone, but the input is explicit about what it should be so assume user is doing it intentionally; warn and convert
    },
    "mixed_string_and_dt": {
        "input": pd.Series(["2025-11-26 12:00", datetime.datetime(2025,11,26,13)]),
        "expect": "raise"
    },
    "mixed_naive_and_tzaware": {
        "input": pd.Series([datetime.datetime(2025,11,26,12), pd.Timestamp("2025-11-26 13:00", tz=TZ)]),
        "expect": "raise"
    },
    "empty_series": {
        "input": pd.Series([]),
        "expect": "raise"
    },
    "invalid_strings": {
        "input": pd.Series(["not a date", "2025-11-26 13:00"]),
        "expect": "raise"
    }
}

In [14]:
def _ensure_datetime(col, tz):
    """
    Vectorized datetime handling:
    
    Rules:
    - Mixtures of types or naive/tz-aware -> raise.
    - TZ-aware datetime -> must match tz, else raise.
    - Naive datetime -> localize to tz (warn once).
    - Strings with offsets -> must match tz at that datetime, else raise.
    - Strings without offsets -> localize to tz (warn).
    
    Returns: pd.Series of tz-aware datetimes in tz.
    """
    # First check for mixed data types
    inferred_type = infer_dtype(col)
    if "mixed" in inferred_type:
        raise ValueError("Ambiguous input: mixture of text and datetime objects.")

    # Next, check if col is already a datetime
    if is_datetime64_any_dtype(col):
        # Check if it's timezone-aware
        if col.dt.tz is not None:
            # Check if it matches the target timezone
            if str(col.dt.tz) == str(tz):
                return col
            else:
                raise ValueError(f"Datetime column has timezone {col.dt.tz} but expected {tz}")
        else:
            # Localize to tz
            warnings.warn(
                "Datetime input has no explicit timezone; assuming it's in "
                f"'{tz}' and localizing accordingly.",
                UserWarning,
            )
            return col.dt.tz_localize(tz)
    elif is_string_dtype(col):
        _OFFSET_RE = re.compile(r'(?:Z|[+-]\d{2}:?\d{2})$')

        # Check for mixed naive/aware using string patterns
        str_view = col.astype("string").fillna("")
        has_offset = str_view.str.match(r'.*' + _OFFSET_RE.pattern)
        
        if has_offset.any() and not has_offset.all():
            raise ValueError(
                f"Datetime input contains a mix of offset-aware and naive strings. "
                "All timestamps must be consistently naive or aware."
            )

        # Parse the strings to datetime objects
        dt_series = pd.to_datetime(col)
        
        # Check if the series has timezone info
        if dt_series.dt.tz is None:
            # Naive datetime - localize to target timezone (assume already in that timezone)
            try:
                result = dt_series.dt.tz_localize(tz, ambiguous='infer')
            except AmbiguousTimeError as e:
                raise ValueError(
                    f"Ambiguous times detected in datetime input that cannot be inferred. "
                    f"This typically means the data is not sorted chronologically. "
                    f"Original error: {e}"
                )
            except NonExistentTimeError as e:
                raise ValueError(
                    f"Non-existent times detected in datetime input. "
                    f"This occurs during 'spring forward' DST transitions when an hour is skipped. "
                    f"Original error: {e}"
                )
            
            warnings.warn(
                "Datetime input has no explicit timezone; assuming it's in "
                f"'{tz}' and localizing accordingly.",
                UserWarning,
            )
            return result
        else:
            # All aware - check for mixed timezones (excluding DST variations)
            tz_objects = dt_series.apply(lambda x: str(x.tzinfo) if x.tzinfo else None)
            unique_tz_objects = tz_objects.unique()
            
            if len(unique_tz_objects) > 1:
                raise ValueError(
                    f"Mixed timezones detected in datetime input: {list(unique_tz_objects)}. "
                    f"All timestamps must be in the same timezone."
                )
            
            # Convert to target timezone
            print(f"Converting from UTC offset {dt_series.dt.tz} to {tz}")
            return dt_series.dt.tz_convert(tz)

In [ ]:
from pandas.testing import assert_series_equal

def series_equal(s1, s2):
    """Check if two series are equal, handling timezone differences."""
    try:
        # Check if both are datetime with timezones
        if hasattr(s1.dtype, 'tz') and hasattr(s2.dtype, 'tz'):
            # Convert both to UTC to normalize timezone objects
            assert_series_equal(s1.dt.tz_convert('UTC'), s2.dt.tz_convert('UTC'))
        else:
            assert_series_equal(s1, s2)
        return True
    except (AssertionError, Exception):
        return False

for test_name, test_data in tests.items():
    try:
        result = _ensure_datetime(test_data["input"], TZ)
        if test_data["expect"] == "ok" and series_equal(result,check):
            print(f"{test_name} passed, output successful.")
        elif test_data["expect"] == "raise" and series_equal(result,check):
            print(f"{test_name} failed - epected error, but output matches target.")
        elif test_data["expect"] == "ok":
            print(f"{test_name} failed - output does not match target.\nOutput: {result}\nExpected: {check}")
        elif test_data["expect"] == "raise":
            print(f"{test_name} failed - epected error, but got output that doesn't match target.")
        else:
            print("Shouldn't be able to get here!")
    except ValueError as e:
        if test_data["expect"] == "ok":
            print(f"{test_name} failed - expected success but got error: {e}")
        elif test_data["expect"] == "raise":
            print(f"{test_name} passed, error raised as expected: {e}")
        else:
            print("Shouldn't be able to get here!")
    print("\n")

naive_strings passed, output successful.


naive_datetimes passed, output successful.


tzaware_matching_timestamp passed, output successful.


tzaware_matching_datetime_zoneinfo passed, output successful.


tzaware_different_tz passed, error raised as expected: Datetime column has timezone UTC but expected America/New_York


Converting from UTC offset UTC-05:00 to America/New_York
string_offset_matching passed, output successful.


Converting from UTC offset UTC+01:00 to America/New_York
string_offset_not_matching passed, output successful.


mixed_string_and_dt passed, error raised as expected: Ambiguous input: mixture of text and datetime objects.


mixed_naive_and_tzaware failed - epected error, but got output that doesn't match target.


empty_series failed - epected error, but got output that doesn't match target.


invalid_strings passed, error raised as expected: Unknown datetime string format, unable to parse: not a date, at position 0




C:\Users\cbrooks\AppData\Local\Temp\ipykernel_19272\1267081005.py:70: UserWarning: Datetime input has no explicit timezone; assuming it's in 'America/New_York' and localizing accordingly.
  warnings.warn(
C:\Users\cbrooks\AppData\Local\Temp\ipykernel_19272\1267081005.py:30: UserWarning: Datetime input has no explicit timezone; assuming it's in 'America/New_York' and localizing accordingly.
  warnings.warn(
C:\Users\cbrooks\AppData\Local\Temp\ipykernel_19272\1267081005.py:70: UserWarning: Datetime input has no explicit timezone; assuming it's in 'America/New_York' and localizing accordingly.
  warnings.warn(
C:\Users\cbrooks\AppData\Local\Temp\ipykernel_19272\1267081005.py:50: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt_series = pd.to_datetime(col)


In [2]:
from src.pharos_ams_query import query_ams_with_basic_auth
from src.process_as_positions import process_daas_cleared_data

In [3]:
token = keyring.get_password("pharos_api", "api_token")
group_start = '2025-03-01'
group_end = '2025-03-02'
most_recent_version = 'true'
mis_report = 'SD_DAASCLEARED'
url = f"https://ams.pharos-ei.com/api/v2/isone/mis/downloads.csv?organization_key=ho-fl&settle_since={group_start}&settle_before={group_end}&most_recent_version={most_recent_version}&report_name={mis_report}"
                
df_raw = query_ams_with_basic_auth(url, token)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_folder = os.path.join(project_root, "data")
mapping_path = os.path.join(data_folder,"maps","ISONE Location Mapping.csv")
# Parse raw MIS data
df_final = process_daas_cleared_data([df_raw], mapping_path)
df_final


0 duplicate rows removed based on most recent Version.


,datetime_he,asset,name,ops_type,service,da_volume,rt_volume,unit,interval_width_s
429,2025-03-01 01:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN 1,Generation,EIR,0,,MW,3600
430,2025-03-01 01:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN 2,Generation,EIR,0,,MW,3600
431,2025-03-01 01:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN 3,Generation,EIR,0,,MW,3600
432,2025-03-01 01:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN 4,Generation,EIR,0,,MW,3600
143,2025-03-01 01:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN 1,Generation,TMNSR,265,,MW,3600
...,...,...,...,...,...,...,...,...,...
285,2025-03-03 00:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN PUMP 3,Pumping,TMNSR,0,,MW,3600
427,2025-03-03 00:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN 1,Generation,TMOR,0,,MW,3600
428,2025-03-03 00:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN PUMP 3,Pumping,TMOR,0,,MW,3600
141,2025-03-03 00:00:00-05:00,Northfield Mountain,NORTHFIELD MOUNTAIN 1,Generation,TMSR,0,,MW,3600
